# Deep Local Learning

Live mode recomputes reduced deep local-learning, distractor, array-scale, and hybrid diagnostics.

### Setup and Dependencies
Imports the trace package, plotting utilities, and configures the default execution mode.


In [ ]:
import os, math, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

RESULT_MODE = "live"
if RESULT_MODE not in {"live", "full_sweep_cache"}:
    raise ValueError("RESULT_MODE must be 'live' or 'full_sweep_cache'")
GREEN, INDIGO, RED, GOLD, GREY, PURPLE, INK = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2", "#b07cc6", "#2b2b2b"

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

def _cache(name):
    print(f"FULL-SWEEP CACHE: {paths.results_dir() / name}")
    return paths.load_result(name)

def _mean_ci(a):
    a = np.asarray(a, float); lo, hi = bootstrap_ci(a)
    return float(a.mean()), lo, hi

print("RESULT_MODE:", RESULT_MODE)
print("data/results:", paths.results_dir())

from mrl_trace.deep import run_deep_local, run_deep_dms, run_array_scale
from mrl_trace.hybrid import run_hybrid_scale

def _series(name, y, min_len=2):
    arr = np.asarray(y, float).ravel()
    if arr.size < min_len or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has insufficient live data for plotting: n={arr.size}")
    return arr

def _values(name, y):
    arr = np.asarray(y, float).ravel()
    if arr.size == 0 or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has no finite live values for plotting")
    return arr

def _smooth(y, win=50):
    arr = _series("curve", y, min_len=2)
    win = int(win)
    if arr.size < max(5, win):
        return arr
    left = win // 2
    right = win - 1 - left
    padded = np.pad(arr, (left, right), mode="edge")
    kernel = np.ones(win, dtype=float) / float(win)
    return np.convolve(padded, kernel, mode="valid")


### Deep Local Learning Performance
Runs the deep local-learning stack against an explicit distractor signal.


In [ ]:
if RESULT_MODE == "live":
    r = run_deep_local(seeds=4, trials=1200)
    src = "LIVE reduced: 4 seeds, 1200 trials"
else:
    r = _cache("exp7_deep_local.npy"); src = "full-sweep cache"
order = [k for k in ["shallow", "elm", "global", "dfa", "no_trace", "dfa_homeo"] if k in r["finals"]]
vals = np.array([np.asarray(r["finals"][k], float).mean() for k in order])
fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 3.8))
for k, c in zip(order, [GREY, GOLD, INDIGO, GREEN, RED, PURPLE]):
    axA.plot(_smooth(r["curves"][k], win=75), color=c, lw=1.5, label=k)
axA.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); axA.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
axA.set_xlabel("trial window"); axA.set_ylabel("reward rate"); axA.set_ylim(0.25, 1.05)
axA.set_title("Deep XOR learning curves"); axA.legend(frameon=False, fontsize=7); _clean(axA)
axB.bar(np.arange(len(order)), vals, color=[GREY, GOLD, INDIGO, GREEN, RED, PURPLE][:len(order)])
axB.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); axB.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
axB.set_xticks(np.arange(len(order))); axB.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
axB.set_ylim(0, 1.05); axB.set_ylabel("final reward rate"); axB.set_title("Final performance")
_clean(axB); fig.suptitle(f"Deep all-local learning [{src}]", fontsize=11); fig.tight_layout(); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("criteria:", r.get("criteria", {}))

### Deep DMS Task
Tests the deep multi-layer network on a Delayed-Match-to-Sample sequence.


In [ ]:
if RESULT_MODE == "live":
    r = run_deep_dms(seeds=4, trials=1200)
    src = "LIVE reduced: 4 seeds, 1200 trials"
else:
    r = _cache("exp13_deep_dms.npy"); src = "full-sweep cache"
order = [k for k in ["dfa", "dfa_dist", "dfa_homeo", "dfa_homeo_dist", "no_trace_dist"] if k in r["finals"]]
vals = np.array([np.asarray(r["finals"][k], float).mean() for k in order])
fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 3.8))
for k, c in zip(order, [INDIGO, GREEN, GOLD, PURPLE, RED]):
    axA.plot(_smooth(r["curves"][k], win=75), color=c, lw=1.5, label=k)
axA.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); axA.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
axA.set_xlabel("trial"); axA.set_ylabel("reward rate"); axA.set_ylim(0.25, 1.05)
axA.set_title("Deep DMS with temporal distractor"); axA.legend(frameon=False, fontsize=7); _clean(axA)
axB.bar(np.arange(len(order)), vals, color=[INDIGO, GREEN, GOLD, PURPLE, RED][:len(order)])
axB.axhline(r.get("chance", 0.5), ls="--", color=RED, lw=1.0); axB.axhline(r.get("crit", 0.75), ls=":", color=GREY, lw=1.0)
axB.set_xticks(np.arange(len(order))); axB.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
axB.set_ylim(0, 1.05); axB.set_ylabel("final reward rate"); axB.set_title("Final performance")
_clean(axB); fig.suptitle(f"Deep temporal-credit diagnostic [{src}]", fontsize=11); fig.tight_layout(); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("criteria:", r.get("criteria", {}))

### Array and Width Scaling
Evaluates the impact of increasing the hidden layer width on the network's resilience.


In [ ]:
if RESULT_MODE == "live":
    r = run_array_scale(H_grid=(8,), p_grid=(0.0, 0.2, 0.5), seeds=2, trials=400, pf_on=False)
    src = "LIVE oriented: H=8, p={0,0.2,0.5}, 2 seeds, 400 trials"
else:
    r = _cache("exp14_array_scale_sweep.npy"); src = "full-sweep cache"
H = np.asarray(r["H"]); p = np.asarray(r["p"], float); grid = np.asarray(r["grid"], float); ctrl = np.asarray(r["ctrl"], float)
fig, ax = plt.subplots(figsize=(6.6, 3.8))
for i, h in enumerate(H):
    ax.plot(p, grid[i, :, 0], "-o", lw=1.6, color=GREEN, label=f"device H={h}" if i == 0 else None)
    ax.plot(p, ctrl[i], "--s", lw=1.2, color=GREY, label=f"no-trace H={h}" if i == 0 else None)
ax.axhline(0.75, ls=":", color=GREY, lw=1.0); ax.axhline(0.5, ls="--", color=RED, lw=1.0)
ax.set_xlabel("fault probability p"); ax.set_ylabel("final reward rate"); ax.set_ylim(0, 1.05)
ax.set_title(f"Array-scale fault diagnostic [{src}]"); ax.legend(frameon=False, fontsize=8); _clean(ax); plt.show()
print("claim status: live-oriented" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("fault prior:", r.get("faults"))

### Hybrid Implementation Test
Checks the performance of a hybrid orientation (proxy front-end).


In [ ]:
if RESULT_MODE == "live":
    r = run_hybrid_scale(H_grid=(16,), p_grid=(0.0, 0.2), seeds=1, trials=200)
    src = "LIVE oriented: proxy front-end, 1 seed, 200 trials"
else:
    r = _cache("tier6_results.npy"); src = "full-sweep cache"
fig, ax = plt.subplots(figsize=(6.4, 3.6))
if "summary" in r:
    labels, vals = [], []
    for key, item in r["summary"].items():
        labels.append(str(key)); vals.append(item[0] if isinstance(item, (tuple, list)) else float(item))
else:
    labels = ["device", "no_trace", "abstract"]
    vals = [np.asarray(r.get("finals", {}).get(k, [np.nan]), float).mean() for k in labels]
ax.bar(np.arange(len(labels)), vals, color=[GREEN, GREY, INDIGO, GOLD][:len(labels)])
ax.axhline(0.5, ls="--", color=RED, lw=1.0); ax.axhline(0.75, ls=":", color=GREY, lw=1.0)
ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("reward rate"); ax.set_ylim(0, 1.05); ax.set_title(f"Hybrid stack diagnostic [{src}]")
_clean(ax); plt.show()
print("claim status: live-oriented" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("front-end:", r.get("front_end", "cached front-end grid"))
# Full-scale regeneration:
# python -m mrl_trace.deep --exp7 --exp13 --exp14 --full
# python -m mrl_trace.hybrid --exp5 --full